In [7]:
# ============================================================
# NEON RUNNER - GOOGLE COLAB VERSION
# Run this ONE cell.
# ============================================================

from IPython.display import HTML, display

game = r'''
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<meta name="viewport"
      content="width=device-width,
               initial-scale=1,
               maximum-scale=1,
               user-scalable=no">

<title>Neon Runner</title>

<style>

* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
    touch-action: none;
}

html, body {
    width: 100%;
    height: 100%;
    overflow: hidden;
    background: #02020a;
    font-family: Arial, sans-serif;
}

#game {
    position: relative;
    width: 100vw;
    height: 100vh;
    overflow: hidden;
    background: linear-gradient(
        #070722,
        #11114a,
        #020208
    );
}

canvas {
    position: absolute;
    left: 0;
    top: 0;
    width: 100%;
    height: 100%;
}

#hud {
    position: absolute;
    z-index: 10;
    top: 15px;
    left: 15px;
    right: 15px;

    display: flex;
    justify-content: space-between;

    color: white;
    font-size: 18px;
    font-weight: bold;
}

#menu,
#gameOver,
#pauseScreen {

    position: absolute;
    z-index: 20;

    inset: 0;

    display: flex;
    flex-direction: column;

    align-items: center;
    justify-content: center;

    text-align: center;

    background: rgba(0,0,15,0.85);

    color: white;
}

.hidden {
    display: none !important;
}

h1 {
    font-size: 52px;

    color: #00ffff;

    text-shadow:
        0 0 10px #00ffff,
        0 0 30px #0088ff;

    margin-bottom: 10px;
}

h2 {
    font-size: 45px;

    color: #ff2875;

    text-shadow:
        0 0 15px #ff2875;
}

button {

    margin-top: 25px;

    padding: 15px 40px;

    border: 2px solid #00ffff;

    border-radius: 12px;

    background: rgba(0,255,255,0.12);

    color: white;

    font-size: 18px;

    font-weight: bold;

    cursor: pointer;

    box-shadow:
        0 0 20px rgba(0,255,255,0.4);
}

button:active {
    transform: scale(0.95);
}

#instructions {
    margin-top: 20px;

    line-height: 1.7;

    color: #bfc8ff;

    font-size: 14px;
}

</style>
</head>

<body>

<div id="game">

<canvas id="canvas"></canvas>

<div id="hud" class="hidden">

    <div>
        Score:
        <span id="score">0</span>
    </div>

    <div>
        Coins:
        <span id="coins">0</span>
    </div>

</div>


<div id="menu">

    <h1>NEON RUNNER</h1>

    <p>Run. Dodge. Survive.</p>

    <button onclick="startGame()">
        PLAY
    </button>

    <p style="margin-top:20px;">
        Best:
        <span id="menuBest">0</span>
    </p>

    <div id="instructions">

        Swipe LEFT / RIGHT → Change lane<br>
        Swipe UP → Jump<br>
        Swipe DOWN → Slide

    </div>

</div>


<div id="gameOver" class="hidden">

    <h2>GAME OVER</h2>

    <p style="margin-top:20px;">
        Score:
        <span id="finalScore">0</span>
    </p>

    <p style="margin-top:10px;">
        Coins:
        <span id="finalCoins">0</span>
    </p>

    <p style="margin-top:10px;">
        Best:
        <span id="finalBest">0</span>
    </p>

    <button onclick="startGame()">
        RUN AGAIN
    </button>

    <button onclick="showMenu()">
        MENU
    </button>

</div>


<div id="pauseScreen" class="hidden">

    <h2>PAUSED</h2>

    <button onclick="resumeGame()">
        RESUME
    </button>

    <button onclick="showMenu()">
        MENU
    </button>

</div>

</div>


<script>

const canvas =
    document.getElementById("canvas");

const ctx =
    canvas.getContext("2d");


let W = window.innerWidth;
let H = window.innerHeight;


function resizeCanvas() {

    W = window.innerWidth;
    H = window.innerHeight;

    const dpr =
        Math.min(
            window.devicePixelRatio || 1,
            2
        );

    canvas.width = W * dpr;
    canvas.height = H * dpr;

    canvas.style.width = W + "px";
    canvas.style.height = H + "px";

    ctx.setTransform(
        dpr,
        0,
        0,
        dpr,
        0,
        0
    );
}


window.addEventListener(
    "resize",
    resizeCanvas
);

resizeCanvas();


/* ==========================================================
   GAME VARIABLES
   ========================================================== */


let running = false;

let paused = false;

let score = 0;

let coins = 0;

let distance = 0;

let speed = 5;

let spawnTimer = 0;

let obstacles = [];

let coinObjects = [];


let best =
    Number(
        localStorage.getItem(
            "neonRunnerBest"
        )
    ) || 0;


document.getElementById(
    "menuBest"
).textContent = best;


/* ==========================================================
   PLAYER
   ========================================================== */


const player = {

    lane: 1,

    x: 0,

    y: 0,

    width: 42,

    height: 65,

    jumpHeight: 0,

    jumpVelocity: 0,

    jumping: false,

    sliding: false,

    slideTimer: 0

};


/* ==========================================================
   LANES
   ========================================================== */


function laneX(lane) {

    const roadWidth =
        Math.min(
            W * 0.75,
            500
        );

    const roadLeft =
        (W - roadWidth) / 2;

    return (
        roadLeft +
        roadWidth *
        ((lane + 0.5) / 3)
    );
}


/* ==========================================================
   START GAME
   ========================================================== */


function startGame() {

    running = true;

    paused = false;

    score = 0;

    coins = 0;

    distance = 0;

    speed = 5;

    spawnTimer = 0;

    obstacles = [];

    coinObjects = [];


    player.lane = 1;

    player.x = laneX(1);

    player.y = H - 150;

    player.jumpHeight = 0;

    player.jumpVelocity = 0;

    player.jumping = false;

    player.sliding = false;


    document
        .getElementById("menu")
        .classList.add("hidden");


    document
        .getElementById("gameOver")
        .classList.add("hidden");


    document
        .getElementById("pauseScreen")
        .classList.add("hidden");


    document
        .getElementById("hud")
        .classList.remove("hidden");


    lastTime = performance.now();

    requestAnimationFrame(gameLoop);
}


/* ==========================================================
   MENU
   ========================================================== */


function showMenu() {

    running = false;

    paused = false;

    document
        .getElementById("menu")
        .classList.remove("hidden");


    document
        .getElementById("gameOver")
        .classList.add("hidden");


    document
        .getElementById("pauseScreen")
        .classList.add("hidden");


    document
        .getElementById("hud")
        .classList.add("hidden");


    document.getElementById(
        "menuBest"
    ).textContent = best;
}


/* ==========================================================
   PAUSE
   ========================================================== */


function togglePause() {

    if (!running) {
        return;
    }

    paused = !paused;

    if (paused) {

        document
            .getElementById("pauseScreen")
            .classList.remove("hidden");

    } else {

        document
            .getElementById("pauseScreen")
            .classList.add("hidden");

        lastTime = performance.now();

        requestAnimationFrame(gameLoop);
    }
}


function resumeGame() {

    if (!running) {
        return;
    }

    paused = false;

    document
        .getElementById("pauseScreen")
        .classList.add("hidden");

    lastTime = performance.now();

    requestAnimationFrame(gameLoop);
}


/* ==========================================================
   PLAYER CONTROLS
   ========================================================== */


function moveLeft() {

    if (!running || paused) {
        return;
    }

    player.lane =
        Math.max(
            0,
            player.lane - 1
        );
}


function moveRight() {

    if (!running || paused) {
        return;
    }

    player.lane =
        Math.min(
            2,
            player.lane + 1
        );
}


function jump() {

    if (
        running &&
        !paused &&
        !player.jumping &&
        !player.sliding
    ) {

        player.jumping = true;

        player.jumpVelocity = -16;
    }
}


function slide() {

    if (
        running &&
        !paused &&
        !player.jumping
    ) {

        player.sliding = true;

        player.slideTimer = 450;
    }
}


/* ==========================================================
   KEYBOARD
   ========================================================== */


document.addEventListener(
    "keydown",
    function(event) {

        if (
            event.key === "ArrowLeft" ||
            event.key.toLowerCase() === "a"
        ) {
            moveLeft();
        }

        if (
            event.key === "ArrowRight" ||
            event.key.toLowerCase() === "d"
        ) {
            moveRight();
        }

        if (
            event.key === "ArrowUp" ||
            event.key === " "
        ) {
            jump();
        }

        if (
            event.key === "ArrowDown" ||
            event.key.toLowerCase() === "s"
        ) {
            slide();
        }

        if (event.key === "Escape") {
            togglePause();
        }

    }
);


/* ==========================================================
   TOUCH
   ========================================================== */


let touchStartX = 0;

let touchStartY = 0;


canvas.addEventListener(
    "touchstart",
    function(event) {

        const touch =
            event.changedTouches[0];

        touchStartX =
            touch.clientX;

        touchStartY =
            touch.clientY;

    },
    { passive: true }
);


canvas.addEventListener(
    "touchend",
    function(event) {

        const touch =
            event.changedTouches[0];

        const dx =
            touch.clientX -
            touchStartX;

        const dy =
            touch.clientY -
            touchStartY;

        const threshold = 35;


        if (
            Math.abs(dx) >
            Math.abs(dy)
        ) {

            if (
                Math.abs(dx) >
                threshold
            ) {

                if (dx > 0) {
                    moveRight();
                } else {
                    moveLeft();
                }

            }

        } else {

            if (
                Math.abs(dy) >
                threshold
            ) {

                if (dy < 0) {
                    jump();
                } else {
                    slide();
                }

            }

        }

    },
    { passive: true }
);


/* ==========================================================
   SPAWN
   ========================================================== */


function spawnObstacle() {

    const lane =
        Math.floor(
            Math.random() * 3
        );


    obstacles.push({

        lane: lane,

        x: laneX(lane),

        y: -80,

        width: 55,

        height: 65

    });
}


function spawnCoin() {

    const lane =
        Math.floor(
            Math.random() * 3
        );


    coinObjects.push({

        lane: lane,

        x: laneX(lane),

        y: -30,

        radius: 12,

        collected: false

    });
}


/* ==========================================================
   UPDATE PLAYER
   ========================================================== */


function updatePlayer(dt) {

    const targetX =
        laneX(player.lane);


    player.x +=
        (
            targetX -
            player.x
        ) *
        Math.min(
            1,
            dt * 12
        );


    if (player.jumping) {

        player.jumpHeight +=
            player.jumpVelocity *
            (dt / 16.67);


        player.jumpVelocity +=
            0.8 *
            (dt / 16.67);


        if (
            player.jumpHeight <= 0
        ) {

            player.jumpHeight = 0;

            player.jumpVelocity = 0;

            player.jumping = false;
        }
    }


    if (player.sliding) {

        player.slideTimer -= dt;

        if (
            player.slideTimer <= 0
        ) {

            player.slideTimer = 0;

            player.sliding = false;
        }
    }
}


/* ==========================================================
   UPDATE OBJECTS
   ========================================================== */


function updateObjects(dt) {

    const movement =
        speed *
        (dt / 16.67);


    obstacles.forEach(
        function(obstacle) {

            obstacle.y +=
                movement * 1.5;

        }
    );


    coinObjects.forEach(
        function(coin) {

            coin.y +=
                movement * 1.5;

        }
    );


    obstacles =
        obstacles.filter(
            function(obstacle) {

                return obstacle.y <
                    H + 100;

            }
        );


    coinObjects =
        coinObjects.filter(
            function(coin) {

                return (
                    coin.y < H + 100 &&
                    !coin.collected
                );

            }
        );
}


/* ==========================================================
   SPAWNING
   ========================================================== */


function updateSpawning(dt) {

    spawnTimer -= dt;


    if (spawnTimer <= 0) {

        spawnObstacle();


        if (
            Math.random() > 0.35
        ) {

            spawnCoin();

        }


        spawnTimer =
            Math.max(
                350,
                900 - speed * 35
            );
    }
}


/* ==========================================================
   DIFFICULTY
   ========================================================== */


function updateDifficulty(dt) {

    distance +=
        speed *
        (dt / 1000);


    score =
        Math.floor(
            distance * 2
        );


    speed +=
        0.0007 * dt;


    speed =
        Math.min(
            speed,
            13
        );
}


/* ==========================================================
   COLLISION
   ========================================================== */


function checkCollisions() {

    const playerHeight =
        player.sliding
            ? player.height * 0.55
            : player.height;


    const py =
        player.y -
        player.jumpHeight;


    for (
        const obstacle
        of obstacles
    ) {

        if (
            obstacle.lane !==
            player.lane
        ) {
            continue;
        }


        const playerLeft =
            player.x -
            player.width / 2;


        const playerRight =
            player.x +
            player.width / 2;


        const playerTop =
            py +
            playerHeight / 2;


        const playerBottom =
            py -
            playerHeight / 2;


        const obstacleLeft =
            obstacle.x -
            obstacle.width / 2;


        const obstacleRight =
            obstacle.x +
            obstacle.width / 2;


        const obstacleTop =
            obstacle.y +
            obstacle.height / 2;


        const obstacleBottom =
            obstacle.y -
            obstacle.height / 2;


        if (
            playerRight >
            obstacleLeft &&

            playerLeft <
            obstacleRight &&

            playerTop >
            obstacleBottom &&

            playerBottom <
            obstacleTop
        ) {

            endGame();

            return;
        }
    }


    for (
        const coin
        of coinObjects
    ) {

        if (
            coin.collected ||
            coin.lane !==
            player.lane
        ) {
            continue;
        }


        const dx =
            player.x -
            coin.x;


        const dy =
            py -
            coin.y;


        const distance =
            Math.sqrt(
                dx * dx +
                dy * dy
            );


        if (distance < 45) {

            coin.collected = true;

            coins++;

            score += 10;
        }
    }
}


/* ==========================================================
   GAME OVER
   ========================================================== */


function endGame() {

    running = false;


    if (score > best) {

        best = score;

        localStorage.setItem(
            "neonRunnerBest",
            best
        );
    }


    document.getElementById(
        "finalScore"
    ).textContent = score;


    document.getElementById(
        "finalCoins"
    ).textContent = coins;


    document.getElementById(
        "finalBest"
    ).textContent = best;


    document
        .getElementById("gameOver")
        .classList.remove("hidden");


    document
        .getElementById("hud")
        .classList.add("hidden");
}


/* ==========================================================
   DRAW BACKGROUND
   ========================================================== */


function drawBackground() {

    const gradient =
        ctx.createLinearGradient(
            0,
            0,
            0,
            H
        );


    gradient.addColorStop(
        0,
        "#070722"
    );


    gradient.addColorStop(
        0.5,
        "#11113b"
    );


    gradient.addColorStop(
        1,
        "#020208"
    );


    ctx.fillStyle =
        gradient;


    ctx.fillRect(
        0,
        0,
        W,
        H
    );


    /* stars */

    ctx.fillStyle =
        "rgba(255,255,255,0.6)";


    for (
        let i = 0;
        i < 45;
        i++
    ) {

        const x =
            (i * 97) % W;


        const y =
            (i * 173) %
            (H * 0.55);


        ctx.fillRect(
            x,
            y,
            2,
            2
        );
    }
}


/* ==========================================================
   DRAW ROAD
   ========================================================== */


function drawRoad() {

    const roadWidth =
        Math.min(
            W * 0.75,
            500
        );


    const roadLeft =
        (W - roadWidth) / 2;


    ctx.fillStyle =
        "#08080f";


    ctx.beginPath();

    ctx.moveTo(
        W * 0.35,
        0
    );

    ctx.lineTo(
        W * 0.65,
        0
    );

    ctx.lineTo(
        roadLeft +
        roadWidth,
        H
    );

    ctx.lineTo(
        roadLeft,
        H
    );

    ctx.closePath();

    ctx.fill();


    ctx.strokeStyle =
        "rgba(0,255,255,0.35)";


    ctx.lineWidth = 2;


    for (
        let lane = 1;
        lane < 3;
        lane++
    ) {

        const topX =
            W *
            (0.35 +
             lane * 0.10);


        const bottomX =
            roadLeft +
            roadWidth *
            (lane / 3);


        ctx.beginPath();

        ctx.moveTo(
            topX,
            0
        );

        ctx.lineTo(
            bottomX,
            H
        );

        ctx.stroke();
    }
}


/* ==========================================================
   DRAW PLAYER
   ========================================================== */


function drawPlayer() {

    const y =
        player.y -
        player.jumpHeight;


    const h =
        player.sliding
            ? player.height * 0.55
            : player.height;


    ctx.save();


    ctx.shadowColor =
        "#00ffff";


    ctx.shadowBlur = 20;


    ctx.fillStyle =
        "#00ffff";


    ctx.fillRect(

        player.x -
        player.width / 2,

        y -
        h / 2,

        player.width,

        h
    );


    ctx.fillStyle =
        "white";


    ctx.fillRect(

        player.x - 10,

        y -
        h / 2 +
        8,

        20,

        12
    );


    ctx.restore();
}


/* ==========================================================
   DRAW OBSTACLES
   ========================================================== */


function drawObstacles() {

    for (
        const obstacle
        of obstacles
    ) {

        ctx.save();


        ctx.shadowColor =
            "#ff1744";


        ctx.shadowBlur = 20;


        ctx.fillStyle =
            "#ff1744";


        ctx.fillRect(

            obstacle.x -
            obstacle.width / 2,

            obstacle.y -
            obstacle.height / 2,

            obstacle.width,

            obstacle.height
        );


        ctx.restore();
    }
}


/* ==========================================================
   DRAW COINS
   ========================================================== */


function drawCoins() {

    for (
        const coin
        of coinObjects
    ) {

        if (coin.collected) {
            continue;
        }


        ctx.save();


        ctx.shadowColor =
            "#ffd700";


        ctx.shadowBlur = 20;


        ctx.fillStyle =
            "#ffd700";


        ctx.beginPath();


        ctx.arc(

            coin.x,

            coin.y,

            coin.radius,

            0,

            Math.PI * 2
        );


        ctx.fill();


        ctx.restore();
    }
}


/* ==========================================================
   DRAW
   ========================================================== */


function draw() {

    drawBackground();

    drawRoad();

    drawCoins();

    drawObstacles();

    drawPlayer();
}


/* ==========================================================
   GAME LOOP
   ========================================================== */


let lastTime = 0;


function gameLoop(time) {

    if (
        !running ||
        paused
    ) {
        return;
    }


    const dt =
        Math.min(
            time -
            lastTime ||
            16.67,

            40
        );


    lastTime = time;


    updatePlayer(dt);

    updateObjects(dt);

    updateSpawning(dt);

    updateDifficulty(dt);

    checkCollisions();


    document.getElementById(
        "score"
    ).textContent = score;


    document.getElementById(
        "coins"
    ).textContent = coins;


    draw();


    if (running) {

        requestAnimationFrame(
            gameLoop
        );
    }
}


/* ==========================================================
   INITIALIZE
   ========================================================== */

player.x = laneX(1);

player.y = H - 150;

draw();

</script>

</body>
</html>
'''

display(HTML(game))